# Build the client-level table

Merges client demographics with the experiment assignment (Test/Control) and aggregates visit-level KPIs up to one row per client.

**Per-client KPIs computed**
- `session_per_client` — number of visits the client made
- `max_step_client` — furthest step ever reached across all their visits
- `n_backward_steps` — total backward navigations across all visits
- `completion_rate` — fraction of the client's visits that completed the funnel
- `avg_steps_per_session` — average events per visit
- `avg_error_rate` — average error rate across visits
- `avg_time_to_completion` — average duration of completed visits (NaN if client never completed)

## Load source tables

In [26]:
import pandas as pd

df1 = pd.read_csv('df_demo_clean.csv')
df2 = pd.read_csv('df_final_experiment_clients.csv')
df_events = pd.read_csv('events_kpi_table.csv')
df_visits = pd.read_csv('visits_kpi_table.csv')

## Merge demographics with experiment assignment

Inner join keeps only clients who appear in both tables (i.e., demo info exists *and* they were assigned to a group).

In [27]:
df_clients_roster = df1.merge(df2, on='client_id', how='inner')

In [28]:
# Drop clients without an assigned Variation. They don't belong to either Test or Control
df_clients_roster = df_clients_roster.dropna(subset=['Variation'])

In [29]:
# 'X' is an unclear/edge gender code. Let's fold into 'U' (unknown) for clean grouping later
df_clients_roster['gendr'] = df_clients_roster['gendr'].replace({'X': 'U'})

## Aggregate visit-level KPIs to client level

In [30]:
summary = df_visits.groupby("client_id").agg(
    session_per_client =("visit_id", "count"),
    max_step_client = ("max_step_reached", "max"),
    n_backward_steps = ("n_backward_steps", "sum"),
    completion_rate = ("reached_confirm", 'mean'),
    avg_steps_per_session=("n_steps", "mean"),
    avg_error_rate=("error_rate", "mean")).reset_index()

summary


,client_id,session_per_client,max_step_client,n_backward_steps,completion_rate,avg_steps_per_session,avg_error_rate
0,169,1,4,0,1.000000,5.000000,0.000000
1,336,1,0,0,0.000000,2.000000,0.000000
2,546,1,4,0,1.000000,5.000000,0.000000
3,555,1,4,0,1.000000,5.000000,0.000000
4,647,1,4,0,1.000000,5.000000,0.000000
...,...,...,...,...,...,...,...
120152,9999729,3,4,1,0.333333,3.666667,0.083333
120153,9999768,1,4,3,1.000000,12.000000,0.250000
120154,9999832,1,1,0,0.000000,2.000000,0.000000
120155,9999839,1,4,0,1.000000,6.000000,0.000000


### Average time to completion

Computed separately because it needs to filter to *completed* visits only. Averaging over all visits (including abandoned ones) would be meaningless. Clients who never completed will end up with `NaN` here, which is correct.

In [31]:
time_completion = df_visits[df_visits["reached_confirm"]].groupby("client_id").agg(
    avg_time_to_completion = ("visit_duration_sec", "mean")).reset_index()

time_completion

,client_id,avg_time_to_completion
0,169,213.0
1,546,133.0
2,555,158.0
3,647,377.0
4,722,599.0
...,...,...
81140,9999451,168.0
81141,9999729,75.0
81142,9999768,486.0
81143,9999839,248.0


## Merge KPIs into the client roster

`how='left'` keeps every client in the roster, even if they have no visits or no completions.

In [32]:
summary_table_client = df_clients_roster.merge(summary, on='client_id', how='left').merge(time_completion, on='client_id', how='left')

## Save

In [33]:
summary_table_client.to_csv("client_kpi_table.csv", index=False)